In [0]:
%run "/Workspace/Jurídico/funcao_tratamento_fechamento/common_functions"

In [0]:
dbutils.widgets.text("dt_gerencial", "")

In [0]:
# Caminho das pastas e arquivos

dt_gerencial = dbutils.widgets.get("dt_gerencial")
diretorio_origem = '/Volumes/databox/juridico_comum/arquivos/trabalhista/bases_gerenciais/external/'

arquivo_consolidado = f'TRABALHISTA_GERENCIAL_(CONSOLIDADO)-{dt_gerencial}.xlsx'

path_consolidado = diretorio_origem + arquivo_consolidado

In [0]:
# Carrega as planilhas em Spark Data Frames
df_consolidado = read_excel(path_consolidado, "'TRABALHISTA'!A6")

In [0]:
df_consolidado = adjust_column_names(df_consolidado)

df_consolidado = remove_acentos(df_consolidado)

In [0]:
lista_datas = []

for cols in df_consolidado.columns:
    if 'DATA' in cols:
        lista_datas.append(cols)

df_consolidado = convert_to_date_format(df_consolidado, lista_datas)

df_consolidado.createOrReplaceTempView('df_consolidado')

In [0]:
df_consolidado_2 = spark.sql('''
    SELECT *
        ,COALESCE(DATA_DE_REGISTRO_DO_ENCERRAMENTO, DATA_DE_ENCERRAMENTO_NO_TRIBUNAL) AS DT_ENCERRAMENTO
    FROM df_consolidado
''')

df_consolidado_2.createOrReplaceTempView('df_consolidado_2')

In [0]:
df_gerencial_resumido = df_consolidado_2.select(
    'PROCESSO_ID',
    'AREA_DO_DIREITO',
    'SUB_AREA_DO_DIREITO',
    'ACAO',
    'GRUPO',
    'DATA_REGISTRADO',
    'STATUS',
    'CARTEIRA',
    'VALOR_DA_CAUSA',
    'PROCESSO_ESTADO',
    'PROCESSO_COMARCA',
    'DISTRIBUICAO',
    'NUMERO_DO_PROCESSO',
    'PARTE_CONTRARIA_NOME',
    'ESCRITORIO',
    'CLASSIFICACAO',
    'PARTE_CONTRARIA_CARGO_CARGO_GRUPO',
    'PARTE_CONTRARIA_CARGO_CARGO_SUB_GRUPO',
    'PARTE_CONTRARIA_CPF',
    'DATA_DE_ENCERRAMENTO_NO_TRIBUNAL',
    'DATA_DE_REGISTRO_DO_ENCERRAMENTO',
    'DATA_DE_SOLICITACAO_DE_ENCERRAMENTO',
    'DT_ENCERRAMENTO',
    'MATRICULA',
    'TERCEIRO_PRINCIPAL',
    'NATUREZA_OPERACIONAL',
    'VALOR_PROVISIONADO',
    'ULTIMA_POSICAO_ESTRATEGIA_JUSTIFICATIVA_ACORDO_DEFESA'
)

df_gerencial_resumido.createOrReplaceTempView('df_gerencial_resumido')

In [0]:
df_encerrados_sem_data = spark.sql('''
    SELECT * FROM (
    SELECT *
    ,coalesce(DATA_DE_ENCERRAMENTO_NO_TRIBUNAL,DATA_DE_REGISTRO_DO_ENCERRAMENTO) AS DT_ENCERR
    FROM df_gerencial_resumido
    WHERE AREA_DO_DIREITO = 'TRABALHISTA' AND STATUS = 'ENCERRADO')
    WHERE DT_ENCERR IS NULL
''') 

df_encerrados_sem_data.createOrReplaceTempView('df_encerrados_sem_data')

In [0]:
from pyspark.sql.functions import col

dt_gerencial = dbutils.widgets.get("dt_gerencial")
ano = int(dt_gerencial[:4])
ano_inicio = ano - 4
mes = dt_gerencial[4:6]
data_inicio = f"'{ano_inicio}-{mes}-01'"

print(data_inicio)

df_gerencial_encerrados = spark.sql(f'''
  SELECT * FROM df_gerencial_resumido
    WHERE STATUS = 'BAIXA PROVISORIA' OR DT_ENCERRAMENTO >= {data_inicio}                                 
''')

df_gerencial_encerrados.createOrReplaceTempView('df_gerencial_encerrados')

df_gerencial_encerrados_2 = spark.sql(f'''
  SELECT * FROM df_gerencial_resumido
    WHERE AREA_DO_DIREITO  = 'TRABALHISTA'                                
''')

df_gerencial_encerrados_2.createOrReplaceTempView('df_gerencial_encerrados')

In [0]:
# %sql

# SELECT * from df_gerencial_encerrados where processo_id = '232179'


In [0]:
df_gerencial_resumido_1 = spark.sql('''

    SELECT * FROM (
    SELECT *
    ,CASE
        WHEN DATA_REGISTRADO <= '2021-09-01' THEN 'LEGADO'
        ELSE 'NOVO' END AS NOVO_X_LEGADO
    FROM df_gerencial_encerrados
    WHERE STATUS IN ('ENCERRADO','BAIXA PROVISORIA'))
''')

df_gerencial_resumido_1.createOrReplaceTempView('df_gerencial_resumido_1')


In [0]:
df_gerencial_resumido_2 = spark.sql('''
    SELECT A.*
    ,B.ACORDO
    ,B.CONDENACAO
    ,B.PENHORA
    ,B.PENSAO
    ,B.CUSTAS
    ,B.OUTROS_PAGAMENTOS
    ,B.IMPOSTO
    ,B.GARANTIA
    ,B.TOTAL_PAGAMENTOS
    FROM df_gerencial_resumido_1 A
    LEFT JOIN databox.juridico_comum.tb_trab_pgto_garantias_custas B
    ON A.PROCESSO_ID = B.PROCESSO_ID
''')

df_gerencial_resumido_2.createOrReplaceTempView('df_gerencial_resumido_2')

df_gerencial_resumido_2 = df_gerencial_resumido_2.fillna(0)

df_gerencial_resumido_2.createOrReplaceTempView('df_gerencial_resumido_3')

In [0]:
df_gerencial_resumido_2.count()

In [0]:
df_gerencial_resumido_3 = spark.sql('''
    SELECT * 
    ,ACORDO + CONDENACAO + PENHORA + PENSAO + CUSTAS + OUTROS_PAGAMENTOS + IMPOSTO + GARANTIA AS TOTAL_PAGAMENTOS_CUSTAS
    FROM df_gerencial_resumido_3
''')

df_gerencial_resumido_3.createOrReplaceTempView('df_gerencial_resumido_3')

In [0]:
# %sql

# SELECT * FROM df_gerencial_resumido_3
# WHERE PROCESSO_ID = 891475


In [0]:
df_gerencial_resumido_4 = spark.sql('''
    SELECT * FROM df_gerencial_resumido_3
    WHERE TOTAL_PAGAMENTOS >= 1
''')

df_gerencial_resumido_4.createOrReplaceTempView('df_gerencial_resumido_4')

In [0]:
df_gerencial_resumido_5 = spark.sql('''
    SELECT *,
        CASE WHEN ACAO IN (
            'AUTO DE INFRAÇÃO',
            'AÇÃO CAUTELAR',
            'AÇÃO DE CONSIGNAÇÃO EM PAGAMENTO',
            'AÇÃO DE CUMPRIMENTO',
            'AÇÃO DE EXECUÇÃO FISCAL',
            'AÇÃO RESCISÓRIA',
            'AÇÃO REVISIONAL',
            'EMBARGOS DE TERCEIRO',
            'HOMOLOGAÇÃO DA TRANSAÇÃO EXTRAJUDICIAL',
            'MANDADO DE SEGURANÇA',
            'PROCESSO ADMINISTRATIVO',
            'PROTESTO JUDICIAL',
            'RECLAMAÇÃO CORREICIONAL',
            'RECLAMAÇÃO TRABALHISTA',
            'TAC - NOT. RECOMENDATÓRIA'
        )
        THEN 'ACAO_VALIDA'
        ELSE 'ACAO_EXCLUIDA' END AS CLASS_ACAO
    FROM df_gerencial_resumido_4
    ''')

df_gerencial_resumido_5.createOrReplaceTempView('df_gerencial_resumido_5')

In [0]:
from pyspark.sql.functions import col, expr

q1, q3 = df_gerencial_resumido_5.approxQuantile("TOTAL_PAGAMENTOS", [0.25, 0.75], 0.01)
iqr = q3 - q1

limite_inferior_1e5 = q1 - 1.5 * iqr
limite_superior_1e5 = q3 + 1.5 * iqr

limite_inferior_5 = q1 - 5 * iqr
limite_superior_5 = q3 + 5 * iqr

print(f'IQR de: {iqr}')
print(f'Na tolerencia de 1.5: Inferior: {limite_inferior_1e5} e superior: {limite_superior_1e5}')
print(f'Na tolerencia de 5.0: Inferior: {limite_inferior_5} e superior: {limite_superior_5}')

df_gerencial_resumido_6 = df_gerencial_resumido_5.withColumn(
    "OUTLIER_1e5",
    (col("TOTAL_PAGAMENTOS") < limite_inferior_1e5) | (col("TOTAL_PAGAMENTOS") > limite_superior_1e5)
).withColumn(
    "OUTLIER_5",
    (col("TOTAL_PAGAMENTOS") < limite_inferior_5) | (col("TOTAL_PAGAMENTOS") > limite_superior_5)
)

df_gerencial_resumido_6.createOrReplaceTempView('df_gerencial_resumido_6')

In [0]:


df_gerencial_resumido_7 = spark.sql(f'''select * from df_gerencial_resumido_6
WHERE STATUS = 'BAIXA PROVISORIA' OR DT_ENCERRAMENTO >= {data_inicio}''')

df_gerencial_resumido_7.createOrReplaceTempView('df_gerencial_resumido_7')

In [0]:
# df_gerencial_resumido_6.count()

In [0]:
import pandas as pd
import re
from shutil import copyfile
import os

# Converter PySpark DataFrame para Pandas DataFrame

pandas_df = df_gerencial_resumido_7.toPandas()

# Salvar o DataFrame com os nomes corrigidos em um arquivo Excel no disco local primeiro
local_path = '/local_disk0/tmp/df_gerencial_resumido.xlsx'
pandas_df.to_excel(local_path, index=False, sheet_name='df_ger_encerrados')

# Definir o nome do arquivo
qtd = 0
path = '/Volumes/databox/juridico_comum/arquivos/modelo_provisao/output/'
list_files = dbutils.fs.ls(path)
nome_arquivo = f'df_ger_encerrados'

# # Essa rotina verifica quantos arquivos com o nome escolhido existem na pasta e faz um controle de versao com base na quantidade
# for file in list_files:
#     name = file.name
#     if nome_arquivo in name:
#         qtd += 1
#         print(f'{name} | {qtd} \n')

# if qtd == 0:
#     print(f'Ainda não foi criado nenhum arquivo com o nome: {nome_arquivo}')
# if qtd >= 1:
#     new_qtd = qtd +1
#     nome_arquivo = f'df_dados_final_{new_qtd}'


# Copiar o arquivo do disco local para o volume desejado
volume_path = f'/Volumes/databox/juridico_comum/arquivos/modelo_provisao/output/{nome_arquivo}.xlsx'
copyfile(local_path, volume_path)

In [0]:
# import pandas as pd
# import re
# from shutil import copyfile
# import os

# # Converter PySpark DataFrame para Pandas DataFrame

# pandas_df = df_encerrados_sem_data.toPandas()

# # Salvar o DataFrame com os nomes corrigidos em um arquivo Excel no disco local primeiro
# local_path = '/local_disk0/tmp/df_encerrados_sem_data.xlsx'
# pandas_df.to_excel(local_path, index=False, sheet_name='df_encerrados_sem_data')

# # Definir o nome do arquivo
# qtd = 0
# path = '/Volumes/databox/juridico_comum/arquivos/modelo_provisao/output/'
# list_files = dbutils.fs.ls(path)
# nome_arquivo = f'df_encerrados_sem_data'

# # # Essa rotina verifica quantos arquivos com o nome escolhido existem na pasta e faz um controle de versao com base na quantidade
# # for file in list_files:
# #     name = file.name
# #     if nome_arquivo in name:
# #         qtd += 1
# #         print(f'{name} | {qtd} \n')

# # if qtd == 0:
# #     print(f'Ainda não foi criado nenhum arquivo com o nome: {nome_arquivo}')
# # if qtd >= 1:
# #     new_qtd = qtd +1
# #     nome_arquivo = f'df_dados_final_{new_qtd}'


# # Copiar o arquivo do disco local para o volume desejado
# volume_path = f'/Volumes/databox/juridico_comum/arquivos/modelo_provisao/output/{nome_arquivo}.xlsx'
# copyfile(local_path, volume_path)